In [58]:
import os
from langchain_ibm import WatsonxEmbeddings
from langchain_ibm import ChatWatsonx
from langchain_classic.agents import create_react_agent, AgentExecutor
from langchain.agents import create_agent
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_ollama import ChatOllama

In [59]:
#.env 내용 가죠오기
load_dotenv()

apikey = os.getenv("WATSONX_API_KEY")
project_id = os.getenv("WATSONX_PROJECT_ID")
watsonx_ai_url = os.getenv("WATSONX_URL")
hf_token = os.getenv("HF_TOKEN")
cohere_api_key = os.getenv("COHERE_API_KEY")
serper_api_key = os.getenv("SERPER_API_KEY")

# HuggingFace model

hugging_llm = ChatOpenAI(
    model="Qwen/Qwen2.5-7B-Instruct:together",
    api_key=hf_token,
    base_url="https://router.huggingface.co/v1",
    temperature= 0
)
# 유료 LLM 선언

watson_llm = ChatWatsonx(
    model_id="ibm/granite-4-h-small",
    url = f"{watsonx_ai_url}",
    api_key = f"{apikey}",
    project_id=f"{project_id}",
    params = {
    "max_tokens": 2000,
    "temperature": 0
    }
)

# 로컬 LLM 선언
qwen_llm = ChatOllama(model="qwen3.5:4b",temperature= 0)

exaone_llm = ChatOllama(model="exaone3.5:2.4b",temperature= 0)

watsonx_enbedding = WatsonxEmbeddings(
    model_id="ibm/granite-embedding-278m-multilingual",
    url = f"{watsonx_ai_url}",
    api_key = f"{apikey}",
    project_id=f"{project_id}"
)

In [7]:
!pip3 install -U ddgs

  Using cached lxml-6.1.1-cp312-cp312-win_amd64.whl.metadata (3.6 kB)
Using cached lxml-6.1.1-cp312-cp312-win_amd64.whl (4.0 MB)
   ---------------------------------------- 0.0/4.7 MB ? eta -:--:--
   ---------------------------------------- 4.7/4.7 MB 31.2 MB/s  0:00:00

   ---------------------------------------- 0/8 [socksio]
   ---------- ----------------------------- 2/8 [lxml]
   ---------- ----------------------------- 2/8 [lxml]
   ---------- ----------------------------- 2/8 [lxml]
   ---------- ----------------------------- 2/8 [lxml]
   ---------- ----------------------------- 2/8 [lxml]
   ---------- ----------------------------- 2/8 [lxml]
   -------------------- ------------------- 4/8 [hpack]
   -------------------- ------------------- 4/8 [hpack]
   ------------------------------ --------- 6/8 [h2]
   ------------------------------ --------- 6/8 [h2]
   ----------------------------------- ---- 7/8 [ddgs]
   ----------------------------------- ---- 7/8 [ddgs]
   --------

### Agent
#### 1. 내장 Tool 사용법

In [ ]:
# search Tool : DuckDuckGoSearch(무료, api 키 발급 x) 

response = watson_llm.invoke("2026년 대한민국 대통령은 누구인가요?")
response.content

'2026년 대한민국 대통령에 대한 정보는 현재로서는 제공할 수 없습니다. 대통령 선거는 5년마다 열리며, 다음 선거는 2027년에 예정되어 있습니다. 따라서 2026년에 대한민국 대통령이 누구인지는 선거 결과에 따라 결정될 것입니다.'

In [4]:
response = watson_llm.invoke("2024년 노벨 물리학상 수상자의 나이는 몇 살인가요?")
response.content

'죄송하지만, 2024년 노벨 물리학상 수상자의 나이를 알려드릴 수 없습니다. 이는 2024년이 아직 끝나지 않았기 때문입니다. 노벨상 수상자는 매년 10월에 발표되며, 수상자의 나이는 그 시점에서 확인할 수 있습니다.'

In [13]:
from langchain_community.tools import DuckDuckGoSearchRun

tools = DuckDuckGoSearchRun()

agent = create_agent(
    model=watson_llm,
    tools=[tools],
    system_prompt="You are a helpful assistant"
)

result =  agent.invoke(
    {
        "messages":[
            {
                "role":"user",
                "content":"2024년 노벨 물리학상 수상자의 나이는 몇 살인가요?"
            }
        ]
    }
)


In [ ]:
from langchain_community.tools import DuckDuckGoSearchRun

tools = DuckDuckGoSearchRun()

agent = create_agent(
    model=watson_llm,
    tools=[tools],
    system_prompt="You are a helpful assistant"
)

result =  agent.invoke(
    {
        "messages":[
            {
                "role":"user",
                "content":"2026년 대한민국 대통령은 누구인가요?"
            }
        ]
    }
)


In [18]:
def print_result(result):
    for i, msg in enumerate(result['messages']):
        print(f"\n==== {i} ====")
        print(type(msg).__name__)

        
        
        if hasattr(msg, "content") and msg.content:
            print(f"Content: {msg.content}")
            
        # 2. LLM이 도구(Tool) 호출을 요청한 경우 (tool_calls 오타 수정 및 조건 추가)
        if hasattr(msg, "tool_calls") and msg.tool_calls:
            print("🔧 Tool Calls Detected:")
            for tool_call in msg.tool_calls:
                print(f"  - Tool Name: {tool_call.get('name')}")
                print(f"    Arguments: {tool_call.get('args')}")
                print(f"    Call ID: {tool_call.get('id')}")

print_result(result)


==== 0 ====
HumanMessage
Content: 2024년 노벨 물리학상 수상자의 나이는 몇 살인가요?

==== 1 ====
AIMessage
🔧 Tool Calls Detected:
  - Tool Name: duckduckgo_search
    Arguments: {'query': '2024 Nobel Prize in Physics laureates age'}
    Call ID: chatcmpl-tool-a98a630c50de5193

==== 2 ====
ToolMessage
Content: 7일 전 · Between 1901 and 2025, 98 Americans have been awarded with the Nobel Prize in physics. There have furthermore been 28 Nobel Prize laureates in physics from ... 2026. 1. 3. · The 2024 Nobel Prize in Physics was awarded jointly to John J. Hopfield and ... Malala Yousafzai became the youngest person to win the Nobel Peace Prize at the ... 2025. 10. 7. · 2024: John Hopfield and Geoffrey Hinton won the 2024 prize for their work on artificial neural networks and the algorithms that enable machines to learn. Their ... 2025. 6. 19. · The Nobel Prize in Physics has been awarded annually since 1901; 226 individuals have received the prize as of 2024; First winner: Wilhelm Conrad Röntgen ... 2025. 10. 

- 구글 웹서치
    - http://serper.dev/ 회원가입 -> API Key 발급
    

In [40]:
from langchain_community.utilities import GoogleSerperAPIWrapper
from langchain_core.tools import Tool

search = GoogleSerperAPIWrapper()

search_tool = Tool(
    name="google_search",
    func=search.run,
    description="""
    Google  검색으로 최신 정보를 찾습니다.
    뉴스, 트렌드 최신 기술 동향 검색에 사용하세요.
    """
)
agent = create_agent(
    model=hugging_llm,
    tools=[search_tool],
    system_prompt="You are a helpful assistant"
)

result =  agent.invoke(
    {
        "messages":[
            {
                "role":"user",
                "content":"2024년 노벨 물리학상 수상자의 나이는 몇 살인가요?"
            }
        ]
    }
)

In [38]:
agent = create_agent(
    model=hugging_llm,
    tools=[search_tool],
    system_prompt="You are a helpful assistant"
)

result =  agent.invoke(
    {
        "messages":[
            {
                "role":"user",
                "content":"2026년 대한민국 대통령은 누구인가요?"
            }
        ]
    }
)

In [41]:
print_result(result)


==== 0 ====
HumanMessage
Content: 2024년 노벨 물리학상 수상자의 나이는 몇 살인가요?

==== 1 ====
AIMessage
Content: 현재 2024년 노벨 물리학상 수상자에 대한 정보는 없습니다. 노벨상은 매년 10월 중순경에 발표되므로, 아직 공식 발표가 이루어지지 않았습니다. 최근의 수상자 정보를 제공해 드리겠습니다.

최근 2023년 노벨 물리학상 수상자의 나이를 확인해 보겠습니다.

🔧 Tool Calls Detected:
  - Tool Name: google_search
    Arguments: {'__arg1': '2023 노벨 물리학상 수상자 나이'}
    Call ID: call_iquhshpqgb6yvpoba4hwf1fe

==== 2 ====
ToolMessage
Content: 스웨덴 왕립과학원 노벨상위원회는 3일 2023년 노벨물리학상 수상자로 피에르 아고스티니 미국 오하이오주립대 교수, 페렌츠 크라우스 독일막스플랑크양자광학연구소·뮌헨대 교수(61), 앤 륄리에 스웨덴 룬드대 교수(65)를 선정했다고 발표했다.

==== 3 ====
AIMessage
Content: 2023년 노벨 물리학상 수상자들의 나이는 다음과 같습니다:

- 피에르 아고스티니: 61세
- 페렌츠 크라우스: 61세
- 앤 륄리에: 65세


- Wikipedia

In [42]:
!pip3 install wikipedia

  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Created wheel for wikipedia: filename=wikipedia-1.4.0-py3-none-any.whl size=11785 sha256=47bad6d8ee7937c798b477418fc8242d34b704f13a3331581d61e502ae3171fe
  Stored in directory: c:\users\soldesk\appdata\local\pip\cache\wheels\63\47\7c\a9688349aa74d228ce0a9023229c6c0ac52ca2a40fe87679b8
Successfully built wikipedia


In [61]:
from langchain_community.utilities import WikipediaAPIWrapper
from langchain_community.tools import WikipediaQueryRun

tools = WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper())

agent = create_agent(
    model=qwen_llm,
    tools=[tools],
    system_prompt="You are a helpful assistant"
)

result =  agent.invoke(
    {
        "messages":[
            {
                "role":"user",
                "content":"앨런 튜링은 누구인가요? 한국어로 대답해주세요"
            }
        ]
    }
)
print_result(result)


==== 0 ====
HumanMessage
Content: 앨런 튜링은 누구인가요? 한국어로 대답해주세요

==== 1 ====
AIMessage
Content: 앨런 튜링 (Alan Turing) 은 1912 년 영국에서 태어난 수학자이자 컴퓨터 과학자로, 현대 컴퓨팅 이론의 아버지로 평가받는 인물입니다. 주요 업적은 다음과 같습니다:

**생애와 배경:**
- 영국의 브리스톨 출신으로, 케임브리지 대학교를 졸업했습니다.
- 2 차 세계대전 당시 영국 해군에서 암호학 연구에 참여했으며, 독일의 Enigma 기계를 해독하는 데 결정적인 역할을 했습니다.

**주요 업적:**
1. **컴퓨터 과학 이론**: 튜링 기계 (Turing Machine) 를 제안하여 현대 컴퓨터의 기본 원리를 정립했습니다.
2. **인공지능 연구**: 1950 년에 "Computing Machinery and Intelligence"라는 논문에서 인공지능이 존재할 수 있다는 논의를 시작했으며, 이후 '튜링 테스트'를 제시했습니다.
3. **암호학**: Bletchley Park 에서 독일의 Enigma 암호 해독을 주도하여 연합군의 승리에 크게 기여했습니다.

**사망:**
- 1952 년에 동성애 혐의로 기소되어 유죄 판결을 받고, 화학 약품으로 자살한 것으로 알려져 있습니다 (실제로는 스스로를 죽였다는 설이 지배적입니다).

튜링은 과학과 기술의 발전에 지대한 공헌을 한 인물로 평가받으며, 1954 년 영국에서 가장 위대한 수학자로 선정되었습니다.


- 파일관련 tool

In [66]:
from langchain_community.tools import ListDirectoryTool, ReadFileTool, WriteFileTool, CopyFileTool, DeleteFileTool

tools = [ReadFileTool(),WriteFileTool(),CopyFileTool(),ListDirectoryTool(), DeleteFileTool()]
agent = create_agent(
    model=watson_llm,
    tools=tools
)

result = agent.invoke(
    {
        "messages":[
            {
                "role":"user",
                # "content":"현재 파일 목록 보여줘"
                # "content":"food3.py 파일의 내용 보여줘"
                # "content":"현재 파일목록을 summary.text로 생성해줘"
                "content":"현재 파일목록을 summary2.text로 생성하고 복사해서 summary.csv파일로 저장후 summary.text 삭제해줘"
            }
        ]
    }
)

print_result(result)


==== 0 ====
HumanMessage
Content: 현재 파일목록을 summary2.text로 생성하고 복사해서 summary.csv파일로 저장후 summary.text 삭제해줘

==== 1 ====
AIMessage
🔧 Tool Calls Detected:
  - Tool Name: list_directory
    Arguments: {'dir_path': '.'}
    Call ID: chatcmpl-tool-9258cb278f8c51b5

==== 2 ====
ToolMessage
Content: .env
.venv
data
db
finder.py
food.py
food2.py
food3.py
langachain3.ipynb
langchain-agent.ipynb
langchain.ipynb
langchain2.ipynb
news.py
news2.py
ollama.ipynb
ollama.js
pdfrag.py
pdf_분쟁조정.ipynb
rag_company.py
rag_naver.ipynb
review.py
summary.txt

==== 3 ====
AIMessage
🔧 Tool Calls Detected:
  - Tool Name: write_file
    Arguments: {'file_path': 'summary2.txt', 'text': '.env\n.venv\ndata\ndb\nfinder.py\nfood.py\nfood2.py\nfood3.py\nlangachain3.ipynb\nlangchain-agent.ipynb\nlangchain.ipynb\nlangchain2.ipynb\nnews.py\nnews2.py\nollama.ipynb\nollama.js\npdfrag.py\npdf_부잡조정.ipynb\nrag_company.py\nrag_naver.ipynb\nreview.py\nsummary.txt'}
    Call ID: chatcmpl-tool-854b49b930d6aaf5

==== 4 ====
ToolMessa